In [ ]:
# Langchain dependencies
from langchain.document_loaders.pdf import PyPDFDirectoryLoader # Importing PDF loader from Langchain
from langchain.text_splitter import RecursiveCharacterTextSplitter # Importing text splitter from Langchain
from langchain.embeddings import OpenAIEmbeddings # Importing OpenAI embeddings from Langchain
from langchain.schema import Document # Importing Document schema from Langchain
from langchain.vectorstores.chroma import Chroma # Importing Chroma vector store from Langchain
from dotenv import load_dotenv # Importing dotenv to get API key from .env file
from langchain.chat_models import ChatOpenAI # Import OpenAI LLM
import os # Importing os module for operating system functionalities
import shutil # Importing shutil module for high-level file operations

In [ ]:
# Directory of HW and Past Essays:
DATA_PATH = "/data/"

# Load Retrival Documents
def load_documents():
  """
  Load PDF documents from the specified directory using PyPDFDirectoryLoader.
  Returns:
  List of Document objects: Loaded PDF documents represented as Langchain
                                                          Document objects.
  """
  # Initialize PDF loader with specified directory
  document_loader = PyPDFDirectoryLoader(DATA_PATH) 
  # Load PDF documents and return them as a list of Document objects
  return document_loader.load() 

# Test load_documents() & Inspect the contents /metadata of the first doc
documents = load_documents() # Call the function
print(documents[0])

In [ ]:
# Chunking
def split_text(documents: list[Document]):
  """
  Split the text content of the given list of Document objects into smaller chunks.
  Args:
    documents (list[Document]): List of Document objects containing text content to split.
  Returns:
    list[Document]: List of Document objects representing the split text chunks.
  """
  # Initialize text splitter with specified parameters
  text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100, # Size of each chunk in characters
    chunk_overlap=20, # Overlap between consecutive chunks
    length_function=len, # Function to compute the length of the text
    add_start_index=True, # Flag to add start index to each chunk
  )

  # Split documents into smaller chunks using text splitter
  chunks = text_splitter.split_documents(documents)
  print(f"Split {len(documents)} documents into {len(chunks)} chunks.")

  # Print example of page content and metadata for a chunk
  document = chunks[0]
  print(document.page_content)
  print(document.metadata)

  return chunks # Return the list of split text chunks

In [ ]:
# Path to the directory to save Chroma database
CHROMA_PATH = "chroma"
def save_to_chroma(chunks: list[Document]):
  """
  Save the given list of Document objects to a Chroma database.
  Args:
  chunks (list[Document]): List of Document objects representing text chunks to save.
  Returns:
  None
  """

  # Clear out the existing database directory if it exists
  if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)

  # Create a new Chroma database from the documents using OpenAI embeddings
  db = Chroma.from_documents(
    chunks,
    OpenAIEmbeddings(),
    persist_directory=CHROMA_PATH
  )

  # Persist the database to disk
  db.persist()
  print(f"Saved {len(chunks)} chunks to {CHROMA_PATH}.")

In [ ]:
def generate_data_store():
  """
  Function to generate vector database in chroma from documents.
  """
  documents = load_documents() # Load documents from a source
  chunks = split_text(documents) # Split documents into manageable chunks
  save_to_chroma(chunks) # Save the processed data to a data store

In [ ]:
# Load environment variables from a .env file
load_dotenv()
# Generate the data store
generate_data_store()

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel
from langchain_google_genai import ChatGoogleGenerativeAI  # For accessing Google AI Gemini

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# set up mistakes format and retrieve parser
class MistakesReport(BaseModel):
    mistakes: list[dict]
    mistake_types: list

error_checker_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a language error analyst. Given an essay, identify all 
    grammatical, syntactic, vocabulary, and register errors. Return a JSON object with:
    mistakes: list of {{type, example, correction}}, and mistake_types: a list of 
    unique mistake category strings. Respond ONLY with a valid JSON."""),
    ("human", "Essay to analyze:\n\n{essay}")
])

# Parse the output of an LLM call to a JSON object
model_w_json_output = model.with_structured_output(MistakesReport)
error_checker_chain = error_checker_prompt | model_w_json_output

# get mistake list JSON
mistakes = await error_checker_chain.ainvoke({"essay": student_essay}) # need to get student essay

In [ ]:
from langchain.agents import create_agent  
from langchain_google_genai import ChatGoogleGenerativeAI  # For accessing Google AI Gemini
from langchain.messages import HumanMessage
from langchain.tools import tool

# general agent human message prompt
mistake_input = f"Here are the student essay mistakes: {mistakes}."


# create subagents & callback
historian_prompt = """
    You are literay analyst specialized in russian essays. Given a list of mistakes and context embeddings of past student work,
    retrieve releavent mistakes and recurring mistake patterns found in the context. Note the frequency of mistakes in the context
    or if they are not seen before in past work.
"""

historian_agent = create_agent(
    model=model,
    system_prompt=historian_prompt,
)

@tool
def call_historian(query: str) -> str:
    """Call historian in order to retrive information on past work in chromadb"""
    # YOU MUST - Use same embedding function as before
    embedding_function = OpenAIEmbeddings()

    # Prepare the database
    db = Chroma(persist_directory=CHROMA_PATH, embedding_function=embedding_function)
    
    # Retrieving the context from the DB using similarity search
    results = db.similarity_search_with_relevance_scores(query, k=3)

    # Check if there are any matching results or if the relevance score is too low
    if len(results) == 0 or results[0][1] < 0.7:
        print(f"Unable to find matching results.")

    # Combine context from matching documents
    context = "\n\n - -\n\n".join([doc.page_content for doc, _score in results])

    
    response = historian_agent.invoke({"messages": [HumanMessage(content=mistake_input+f"Use this as past work context for answering: {context}")]})
    return response["messages"][-1].content


# create main tutor agent
tutor_prompt = """
You are a helpful Russian language tutor for an English speaking student. Given a list of mistakes found in a student's Russian essay, 
you will create a comprenhsive guide for how the student can avoid these mistakes in the future. To create your guide, use the historian 
agent to get information on if the student made these mistakes before, how frequently, or if they are new. And, use the research agent to get resources
that the student can use to work on their mistakes. Your guide should provide more detailed breakdowns for topics of frequent issue.
"""

tutor_agent = create_agent(
    model=model,
    system_prompt=tutor_prompt,
    tools=[call_historian],
)

In [ ]:
# Call main tutor agent and output results
response = tutor_agent.invoke(
    {"messages": [HumanMessage(content=mistake_input)]}
)